In [1]:
import pandas as pd
import numpy as np
%config IPCompliter.greedy=True

In [2]:
train = pd.read_csv("../data/raw/train.csv")
test  = pd.read_csv("../data/raw/test.csv")

In [3]:
train = train.copy()

# target → 0/1
train["Churn"] = (train["Churn"] == "Yes").astype(int)

# новая фича
train["avg_charge"] = train["TotalCharges"] / train["tenure"].replace(0, np.nan)

In [8]:
cat_cols = train.select_dtypes(include=["object", "str"]).columns.tolist()

num_cols = train.select_dtypes(include=["int64", "float64"]).columns.tolist()
num_cols.remove("Churn")
num_cols.remove("id")

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from lightgbm import LGBMClassifier

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ]
)

model = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1
)

pipe = Pipeline([
    ("prep", preprocessor),
    ("model", model)
])

In [12]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

X = train.drop(columns=["Churn"])
y = train["Churn"]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof = np.zeros(len(train))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    pipe.fit(X_tr, y_tr)
    
    oof[va_idx] = pipe.predict_proba(X_va)[:, 1]
    
    print(f"Fold {fold} AUC:", roc_auc_score(y_va, oof[va_idx]))

print("OOF AUC:", roc_auc_score(y, oof))

[LightGBM] [Info] Number of positive: 107054, number of negative: 368301
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.032944 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 922
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 46
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225209 -> initscore=-1.235567
[LightGBM] [Info] Start training from score -1.235567


D:\ml_projects\venvs\churn_s6e3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 1 AUC: 0.9157674652890394
[LightGBM] [Info] Number of positive: 107054, number of negative: 368301
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.094560 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 922
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 46
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225209 -> initscore=-1.235567
[LightGBM] [Info] Start training from score -1.235567


D:\ml_projects\venvs\churn_s6e3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 2 AUC: 0.9168810036005455
[LightGBM] [Info] Number of positive: 107053, number of negative: 368302
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.115794 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 922
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 46
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225206 -> initscore=-1.235579
[LightGBM] [Info] Start training from score -1.235579


...

Fold 3 AUC: 0.9161878523649338
[LightGBM] [Info] Number of positive: 107053, number of negative: 368302
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.105726 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 922
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 46
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225206 -> initscore=-1.235579
[LightGBM] [Info] Start training from score -1.235579


D:\ml_projects\venvs\churn_s6e3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 4 AUC: 0.9173637891530289
[LightGBM] [Info] Number of positive: 107054, number of negative: 368302
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.040710 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 922
[LightGBM] [Info] Number of data points in the train set: 475356, number of used features: 46
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225208 -> initscore=-1.235570
[LightGBM] [Info] Start training from score -1.235570


D:\ml_projects\venvs\churn_s6e3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 5 AUC: 0.9145155237754387
OOF AUC: 0.9161320254397293
